# 04 — Leakage-safe supervised dataset

Create chronological splits first, fit normalization on training only, and then construct temporal windows.

Default task:

\[
X_t=(SST_{t-13},\ldots,SST_t),\qquad y_t=SST_{t+7}.
\]

In [ ]:
from pathlib import Path

import numpy as np
from torch.utils.data import DataLoader

from oisst_fno.data import (
    ForecastSpec,
    SSTWindowDataset,
    Standardizer,
    open_oisst,
    temporal_split,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
path = sorted((ROOT / "data" / "raw").glob("oisst_*_ne_atlantic.nc"))[-1]
sst = open_oisst(path)["sst"]

TRAIN_END = "2024-12-31"
VALIDATION_END = "2025-12-31"
SPEC = ForecastSpec(lookback_days=14, horizon_days=7)

train_da, val_da, test_da = temporal_split(sst, TRAIN_END, VALIDATION_END)
print(train_da.time.values[[0, -1]])
print(val_da.time.values[[0, -1]])
print(test_da.time.values[[0, -1]] if len(test_da.time) else "empty test split")

In [ ]:
scaler = Standardizer.fit(train_da.values)
train_values = scaler.transform(train_da.values)
val_values = scaler.transform(val_da.values)
test_values = scaler.transform(test_da.values) if len(test_da.time) else None

train_ds = SSTWindowDataset(train_values, SPEC)
val_ds = SSTWindowDataset(val_values, SPEC)
test_ds = SSTWindowDataset(test_values, SPEC) if test_values is not None and len(test_values) >= 21 else None

x, y, mask = train_ds[0]
print("x:", tuple(x.shape), "y:", tuple(y.shape), "mask:", tuple(mask.shape))
print("train windows:", len(train_ds), "validation windows:", len(val_ds))

In [ ]:
# For FNO input, concatenate the static ocean mask as an extra channel.
def collate_with_mask(batch):
    import torch
    xs, ys, masks = zip(*batch)
    x = torch.stack(xs)
    y = torch.stack(ys)
    mask = torch.stack(masks)
    x_with_mask = torch.cat((x, mask), dim=1)
    return x_with_mask, y, mask

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_with_mask)
x_batch, y_batch, mask_batch = next(iter(train_loader))
print(x_batch.shape, y_batch.shape, mask_batch.shape)

### Important nuance

`shuffle=True` is valid **inside the already defined training set** for SGD. It is not equivalent to randomly splitting the time series.

For strict forecast-origin evaluation across a split boundary, a later extension can build windows from a shared continuous array while assigning samples by target date. That permits validation/test forecasts to use legitimate historical context from the immediately preceding split without leaking future targets.

## Forecast target dates are part of the dataset contract

Every prediction must retain its target timestamp. Later notebooks use those dates for paired error analysis, season grouping, and validation-defined regime thresholds. Dropping dates from a spatiotemporal benchmark makes leakage and subgroup analysis much harder to audit.

In [ ]:
from oisst_fno.data import forecast_target_times

validation_target_times = forecast_target_times(val_da.time.values, SPEC)
print("first validation target:", validation_target_times[0])
print("last validation target:", validation_target_times[-1])
print("n validation forecasts:", len(validation_target_times))